# Part 4. Delivery Layer and Reliability Perspective

## Part Scope

This section defines how analytical outputs are delivered for review, export, and downstream reuse.

The role of this layer is not limited to visual presentation.  
The same analytical results are also reorganized into downloadable artifacts and queryable storage structures so that the workflow does not end at a single screen-level display.

## Review-Oriented Application Layout

The delivery layer is designed to preserve analytical context during review.

Instead of presenting only final charts or summaries, the interface keeps normalized records, KPI outputs, data-availability boundaries, structured insights, prompt-ready inputs, and stored-query results connected in one review path.

This design allows users to move from source-level records to interpreted output without losing the intermediate structure that supports reliability.

In [ ]:
def _fcf(snapshot):
    ocf = getattr(snapshot, "operating_cash_flow", None)
    capex = getattr(snapshot, "capex", None)
    if ocf is None or capex is None:
        return None
    return ocf - capex


def _get_share_count(snapshot):
    diluted = getattr(snapshot, "shares_outstanding_diluted", None)
    basic = getattr(snapshot, "shares_outstanding_basic", None)
    if diluted not in (None, 0):
        return diluted
    if basic not in (None, 0):
        return basic
    return None


def _eps(snapshot):
    ni = getattr(snapshot, "net_income", None)
    shares = _get_share_count(snapshot)
    if ni is None or shares in (None, 0):
        return None
    return ni / shares


def _bvps(snapshot):
    equity = getattr(snapshot, "total_equity", None)
    shares = _get_share_count(snapshot)
    if equity is None or shares in (None, 0):
        return None
    return equity / shares


def build_snapshot_preview_df(snapshots):
    rows = []
    for s in sorted(snapshots, key=lambda x: x.fiscal_year):
        rows.append(
            {
                "Fiscal Year": s.fiscal_year,
                "Revenue": getattr(s, "revenue", None),
                "Net Income": getattr(s, "net_income", None),
                "Operating Cash Flow": getattr(s, "operating_cash_flow", None),
                "CapEx": getattr(s, "capex", None),
                "FCF": _fcf(s),
                "EPS": _eps(s),
                "BVPS": _bvps(s),
                "Available Fields": getattr(s, "available_fields_count", None),
                "Missing Fields": getattr(s, "missing_fields_count", None),
                "Minimal Validity": getattr(s, "minimal_validity", None),
            }
        )
    return pd.DataFrame(rows)

In [ ]:
snapshot_preview_df = build_snapshot_preview_df(snapshots)
snapshot_preview_df

## Export and Reuse

The delivery layer preserves multiple analysis states as separate exportable artifacts.

- normalized annual snapshots for source-level review
- KPI tables for metric-level reuse
- structured insights for interpretation-level export

By separating these outputs, the workflow keeps intermediate analytical structure visible outside the application instead of collapsing everything into a single final report.

In [ ]:
def build_insights_export_df(insights):
    rows = []
    for ins in insights or []:
        row = {
            "key": ins.get("key"),
            "category": ins.get("category"),
            "title": ins.get("title"),
            "status": ins.get("status"),
            "priority": ins.get("priority"),
            "summary": ins.get("summary"),
            "risk_note": ins.get("risk_note"),
        }
        evidence = ins.get("evidence") or {}
        for k, v in evidence.items():
            row[f"evidence_{k}"] = v
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
from core.sql_store import snapshots_to_dataframe, kpi_metrics_to_dataframe

snapshots_export_df = snapshots_to_dataframe(snapshots=snapshots, ticker=ticker)
kpi_export_df = kpi_metrics_to_dataframe(
    ticker=ticker,
    growth=growth,
    valuation=valuation,
    yoy_list=yoy_list,
)
insights_export_df = build_insights_export_df(insights)

In [ ]:
snapshots_export_df.head(3), kpi_export_df.head(5), insights_export_df.head(5)

## Queryable Storage Design

The delivery layer is also connected to a queryable SQLite backend.

Instead of storing only a final text result, the workflow persists multiple analytical layers separately: run metadata, annual snapshots, KPI metrics, and structured insights.  
This allows the same analysis run to be revisited later through table-based queries rather than through visual inspection alone.

The storage layer therefore extends the system from a display workflow into a reusable analytical record structure.  
The storage schema mirrors the layered workflow itself: run metadata, annual source records, metric outputs, and interpretation objects are persisted separately so that each layer remains independently queryable.

In [ ]:
from core.sql_store import write_analysis_to_sqlite, query_to_dataframe

In [ ]:
db_path = "analysis_store.db"

sql_write_result = write_analysis_to_sqlite(
    db_path=db_path,
    ticker=ticker,
    snapshots=snapshots,
    growth=growth,
    valuation=valuation,
    insights=insights,
    yoy_list=yoy_list,
)

sql_write_result

## SQL Query Layer

The stored results are not treated as passive logs.

The application exposes a query layer that shows sample SQL statements and immediately returns the corresponding result tables inside the same interface.  
This makes the saved outputs inspectable through database queries without requiring a separate analysis environment.

As a result, storage is connected directly back to review.

In [ ]:
from core.sql_store import write_analysis_to_sqlite

db_path = "analysis_store.db"

sql_write_result = write_analysis_to_sqlite(
    db_path=db_path,
    ticker=ticker,
    snapshots=snapshots,
    growth=growth,
    valuation=valuation,
    insights=insights,
    yoy_list=yoy_list,
)

sql_write_result

In [ ]:
from core.sql_store import query_to_dataframe

snapshot_trend_sql = """
SELECT ticker, fiscal_year, revenue, net_income, operating_cash_flow, capex
FROM snapshots
WHERE ticker = ?
ORDER BY fiscal_year
"""

query_to_dataframe(db_path, snapshot_trend_sql, params=[ticker])

## Reliability Perspective

The delivery layer also preserves the reliability boundaries established earlier in the workflow.

The application does not present only headline metrics.  
It also exposes coverage information, missing-field conditions, minimum-validity status, calculability flags, and recent missing-metric information through the data-quality table, KPI metadata, and year-over-year detail.

This matters because display without interpretability boundaries would make the final interface look cleaner while weakening the reliability of the outputs.

In [ ]:
def build_coverage_df(snapshots):
    fields = [
        ("revenue", "Revenue"),
        ("net_income", "Net Income"),
        ("operating_cash_flow", "Operating Cash Flow"),
        ("capex", "CapEx"),
        ("total_equity", "Total Equity"),
        ("current_assets", "Current Assets"),
        ("current_liabilities", "Current Liabilities"),
        ("shares_outstanding_diluted", "Diluted Shares"),
    ]
    total = len(snapshots) if snapshots else 0
    rows = []
    for field_name, label in fields:
        available = sum(getattr(s, field_name, None) is not None for s in snapshots)
        rows.append(
            {
                "Field": label,
                "Available Years": available,
                "Coverage Ratio": (available / total) if total else None,
            }
        )
    return pd.DataFrame(rows)

In [ ]:
coverage_df = build_coverage_df(snapshots)
coverage_df

In [ ]:
reliability_preview = {
    "latest_snapshot_metadata": growth.get("latest_snapshot_metadata"),
    "calculation_flags": growth.get("calculation_flags"),
    "kpi_metadata": growth.get("kpi_metadata"),
}

reliability_preview

## Closing Note

At the end of the workflow, the system no longer depends on a single interpretation path.

The same analytical results can be reviewed visually, exported as structured files, stored as normalized tables, queried through SQL, and extended into constrained narrative output.  
This final layer closes the workflow by turning intermediate calculations into reusable analytical deliverables.